# Proyecto dental CNN en Google Colab

Notebook principal para entrenar con GPU de Colab y generar una radiografia segmentada/rotulada. La metodologia mantiene la idea conversada: CNN para segmentacion semantica, datasets de Kaggle y post-procesamiento para separar instancias y dibujar etiquetas.

Antes de empezar, sube la carpeta `proyecto_dental_cnn` a Google Drive, idealmente en `Mi unidad/proyecto_dental_cnn`.

## 1. Conectar Drive y seleccionar GPU

En Colab: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU`.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/proyecto_dental_cnn')
assert (PROJECT_DIR / 'dental_cnn').exists(), f'No encontre el proyecto en {PROJECT_DIR}'
%cd {PROJECT_DIR}

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Instalar dependencias

Colab ya trae PyTorch con CUDA en la mayoria de runtimes, por eso instalamos solo librerias de datos, imagenes y Kaggle.

In [ ]:
!python -m pip install -q -r requirements_colab.txt

## 3. Descargar datasets

Se descargan los dos datasets definidos originalmente con `kagglehub`.

In [ ]:
!python -m dental_cnn.download_datasets

## 4. Inspeccion rapida del dataset

Esto sirve para mostrar en la presentacion que el entrenamiento usa anotaciones reales y que el dataset grande aporta mas clases/problemas dentales.

In [ ]:
from collections import Counter
from dental_cnn.data import discover_records, resolve_dataset_roots

roots = resolve_dataset_roots([], download=True)
train_records, train_classes = discover_records(roots, 'train', 'auto')
valid_records, valid_classes = discover_records(roots, 'valid', 'auto')
counts = Counter(box.class_name for rec in train_records for box in rec.boxes)

print('Dataset roots:')
for root in roots:
    print(' -', root)
print('Imagenes train:', len(train_records))
print('Imagenes valid:', len(valid_records))
print('Clases:', sorted(set(train_classes) | set(valid_classes)))
print('Top 12 clases train:', counts.most_common(12))

## 5. Entrenar U-Net binaria

Esta es la corrida recomendada para la entrega: usa GPU, mixed precision (`--amp`) y guarda resultados en Drive. La mascara binaria agrupa regiones dentales/anotadas y luego el programa separa instancias con post-procesamiento.

In [ ]:
RUN_DIR = Path('/content/drive/MyDrive/dental_cnn_runs/binary_unet')

!python -m dental_cnn.train --mask-mode binary --epochs 20 --batch-size 16 --image-width 512 --image-height 256 --base-channels 16 --num-workers 2 --amp --output-dir "{RUN_DIR}"

## 6. Entrenamiento alternativo: clases dentales

Opcional. Esta variante entrena salida multiclasica para mostrar clasificacion de hallazgos como caries, implantes, coronas, tratamientos de conducto y dientes impactados. Es mas dificil que la binaria, pero alinea bien con el paper de deteccion/clasificacion.

In [ ]:
# FOCUSED_RUN_DIR = Path('/content/drive/MyDrive/dental_cnn_runs/focused_multiclass')
# !python -m dental_cnn.train \
#   --mask-mode multiclass \
#   --include-class Caries \
#   --include-class Filling \
#   --include-class Implant \
#   --include-class Crown \
#   --include-class 'Root Canal Treatment' \
#   --include-class 'impacted tooth' \
#   --include-class 'Missing teeth' \
#   --epochs 15 \
#   --batch-size 12 \
#   --image-width 512 \
#   --image-height 256 \
#   --base-channels 16 \
#   --num-workers 2 \
#   --amp \
#   --output-dir "{FOCUSED_RUN_DIR}"

## 7. Generar una imagen rotulada de prueba

In [ ]:
from IPython.display import Image, display

sample_image = None
for root in roots:
    candidates = list((root / 'test').glob('*.jpg'))
    if not candidates:
        candidates = list((root / 'COCO' / 'COCO' / 'test').glob('*.jpg'))
    if candidates:
        sample_image = candidates[0]
        break

assert sample_image is not None, 'No encontre imagenes de prueba.'
output_image = Path('/content/drive/MyDrive/dental_cnn_runs/predictions/demo_rotulada.png')

!python -m dental_cnn.predict "{sample_image}" --checkpoint "{RUN_DIR / 'best_model.pt'}" --output "{output_image}" --threshold 0.5 --min-area 120 --label-style fdi
display(Image(filename=str(output_image)))

## 8. Probar con una radiografia propia

Sube una imagen panoramica dental (`.jpg` o `.png`) y el modelo generara la salida segmentada.

In [ ]:
from google.colab import files

uploaded = files.upload()
custom_image = Path(next(iter(uploaded)))
custom_output = Path('/content/drive/MyDrive/dental_cnn_runs/predictions/mi_radiografia_rotulada.png')

!python -m dental_cnn.predict "{custom_image}" --checkpoint "{RUN_DIR / 'best_model.pt'}" --output "{custom_output}" --threshold 0.5 --min-area 120 --label-style fdi
display(Image(filename=str(custom_output)))

## Nota para defender el proyecto

Los datasets publicos usados contienen principalmente cajas de anotacion. Por eso el entrenamiento genera mascaras debiles a partir de las cajas y la etapa de inferencia aplica morfologia/componentes conectados para refinar regiones. Esto calza con la metodologia de CNN + post-procesamiento descrita en la literatura revisada, pero no debe presentarse como diagnostico clinico.